In [10]:
# USGS metadata cleaning – dates & keys
# -------------------------------------------------------------
# This script is meant to run in a Jupyter notebook. You can
# paste cells one-by-one, or run as a .py script. It:
#   - Loads the CSV (no edits to the original file)
#   - Creates a unique key id_page_number
#   - Builds cleaned coordinates (best-available lat/long)
#   - Parses dates_of_recording into year_start / year_end
#   - Enforces the rule: ignore any year > 1980 and any
#     "present/current year/to date" open-ends
#   - Standardizes water_type into a helper column
#   - Exports a non-destructive cleaned copy + a small
#     manual-review file for messy dates
# -------------------------------------------------------------

# %% Imports
import os
import re
import pandas as pd
from typing import Dict, Any, List, Optional, Tuple

# %% User paths (Windows)
DATA_DIR = r"C:\Users\aeliz\Dropbox\Documents\Jupyter Notebooks\chap2_usgs\data"
INPUT_CSV = os.path.join(DATA_DIR, "cleaned_metadata.csv")
# Safe outputs alongside input (non-destructive)
OUTPUT_CSV = os.path.join(DATA_DIR, "cleaned_metadata_with_years.csv")
REVIEW_CSV = os.path.join(DATA_DIR, "dates_needs_review.csv")

# %% Load
# We do not modify the original file. We work in-memory, then save NEW files.
df = pd.read_csv(INPUT_CSV, dtype=str, keep_default_na=False)
print(f"Loaded {len(df):,} rows from {INPUT_CSV}")

# %% Basic hygiene: uniform column names (no changes to originals in file)
# Ensure expected columns exist; if not, raise with a helpful message.
required_cols = [
    'id','page_number','inferred_latitude','inferred_longitude',
    'actual_latitude','actual_longitude','location','townships_ranges_sections',
    'watersource_name','actual_county','inferred_county','dates_of_recording',
    'temporal_resolution','units_of_measurement','water_type','keyterms'
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing expected column(s): {missing}")

# Strip leading/trailing whitespace in all string cells (non-destructive)
df = df.map(lambda x: x.strip() if isinstance(x, str) else x)

Loaded 63,713 rows from C:\Users\aeliz\Dropbox\Documents\Jupyter Notebooks\chap2_usgs\data\cleaned_metadata.csv


In [11]:
# load preview dataframe 

def preview(dataframe, n=50, random=False, start=None, end=None):
    """
    Preview a chunk of a DataFrame nicely formatted in Jupyter.
    - n: number of rows
    - random=True for random sample
    - start/end: row slice
    """
    if random:
        return dataframe.sample(n).style
    elif start is not None and end is not None:
        return dataframe.loc[start:end].style
    else:
        return dataframe.head(n).style

preview(df, n=100)            # first 100 rows
preview(df, random=True, n=20) # 20 random rows
preview(df, start=500, end=520) # rows 500–520

,id,page_number,inferred_latitude,inferred_longitude,actual_latitude,actual_longitude,location,townships_ranges_sections,watersource_name,actual_county,inferred_county,dates_of_recording,temporal_resolution,units_of_measurement,water_type,keyterms
500,3146,259,32.9466,-110.8504,,,"Near Winkelman, Pinal County",[],SAN PEDRO RIVER NEAR WINKELMAN,"Pinal County, Arizona",,10/01/1964-09/30/1965,Daily,{'Specific_Conductance': 'micromhos at 25°C'},Stream Discharge,['Specific conductance']
501,1592,231,38.1833,-120.7167,,,"Below Salt Springs Dam, North Fork of Mokelumne River","['[T. 8 N., R. 16 E., Sec. 33]']",NORTH FORK OF MOKELUMNE RIVER BELOW SALT SPRINGS DAM,Calaveras County,,10/01/1932-09/30/1933,Daily,"{'Discharge': 'second-feet', 'Gage Height': 'feet'}",Stream Discharge,"['Discharge', 'Gage Height', 'Run-off']"
502,614,235,,,37.676111,-121.264167,"Water-stage recorder in El Pescadero Grant at Durham Ferry highway bridge, 3 miles downstream from Stanislaus River and 3.4 miles northeast of Vernalis",[],"San Joaquin River near Vernalis, Calif.",San Joaquin County,,10/01/1947-09/30/1948,Daily,Second-feet,Stream Discharge,"['Discharge', 'San Joaquin River', 'Vernalis', 'stream flow', 'precipitation runoff']"
503,2513,26,,,34.0522,-118.2437,"Los Angeles, California",[],Los Angeles Rainfall,Los Angeles County,,07/01/1877-06/30/1946,Yearly,inches,Precipitation,"['Rainfall', 'Cumulative Departure', '68-year Average']"
504,70038394,49,34.6461,-120.4625,,,"Near Lompoc, California",[],Santa Ynez River Basin,Santa Barbara County,,11/13/1963-09/24/1964,Variable intervals within the range of daily to weekly measurements,"{'Temperature': '°F', 'Discharge': 'cfs (cubic feet per second)', 'Sediment Concentration': 'ppm (parts per million)', 'Suspended Sediment Discharge': 'tons per day'}",Stream Discharge,"['Suspended sediment', 'Sediment discharge', 'Sampling point', 'Discharge (cfs)']"
505,1305,60,34.097,-117.7462,,,"One-fourth mile west of Spadra, Pomona quadrangle",[],S.E. Hicks Well,,Los Angeles County,"Oct. 8, 1904-Jan. 4, 1911",Variable (monthly to yearly),"{'Depth_below_bench_mark': ['Feet', 'Inches']}",Groundwater,"['bored well', 'wind lift', 'domestic use']"
506,70038675,82,,,33.568889,-116.511944,"near Anza, Riverside Pines to County, Palms Highway, in SWINENW sec 18, T.7 and 9.4 miles east of Anza.","['[SWINENW sec 18, T.7]', '9.4 miles east of Anza']",PALM CANYON CREEK TRIBUTARY,San Bernardino County,,10/1969-09/1970,Daily,cubic feet per second (cfs) and inches for precipitation,Stream Discharge,"['tributary', 'stream discharge', 'culvert']"
507,648,413,,,39.5375,-121.3458333,"NW1 sec. 6, T. 19 N., R. 6 E.","['T. 19 N., R. 6 E., NW1 sec. 6']",South Fork Feather River at Enterprise,Plumas,,10/01/1911-09/30/1952,Daily,cubic feet per second (cfs),Stream Discharge,"['stream discharge', 'flow rates', 'gage height']"
508,56361,103,32.7806,-114.9904,32.7767,-115.573,"NORTHERLY INTERNATIONAL BOUNDARY, ABOVE MORELOS DAM, NEAR ANDRADE, CALIF.",,COLORADO RIVER,,Imperial County,10/09/1974-09/23/1975,Weekly,"{'Total_Photo_Phytoplankton_Per': 'ML)', 'Seston_Ash_Weight': 'MG/L', 'Periphyton_Biomass_Ash_Weight': 'G/SQ M', 'Periphyton_Biomass_Total_Dry_Weight': 'G/SQ M', 'Uncorrected_Periphyton_Chlorophyll_A': 'MG/SQ M', 'Chlorophyll_A': 'UG/L', 'Chlorophyll_B': 'UG/L', 'Fecal_Coli-Form_Col.Per': '100 ML)', 'Tococcic_Colonies_Per': '100 ML)', 'Strep': None, 'Total_Organic_Carbon_C': 'MG/L'}",Stream Discharge,"['WATER QUALITY DATA', 'PHYTOPLANKTON', 'SESTON', 'PERIPHYTON BIOMASS', 'CHLOROPHYLL A', 'FECAL COLIFORM']"
509,3090,278,,,40.18666667,-121.18638889,"near Prattville, Calif.","['NW1 sec.22, T.27 N., R.7 E.']",Butt Creek below Almanor-Butt Creek tunnel,Plumas County,Plumas County,10/01/1964-09/30/1965,Daily,cubic feet per second (cfs),Stream Discharge,"['stream discharge', 'cubic feet per second', 'gage height', 'discharge measurements']"


In [12]:
# %% Unique key: id_page_number (keep originals too)
# id may repeat across multiple pages; id+page_number is unique per row

def make_key(row: pd.Series) -> str:
    return f"{row['id']}_{row['page_number']}"

df['id_page_number'] = df.apply(make_key, axis=1)

# %% Coordinates: "best available" lat/long helper while preserving originals
# If actual_* exists and is valid, use it; otherwise fall back to inferred_*

# Convert-friendly helper (does not throw on bad values)
def _to_float_or_none(v: Any) -> Optional[float]:
    try:
        if v == '' or v is None:
            return None
        return float(str(v))
    except Exception:
        return None

actual_lat = df['actual_latitude'].map(_to_float_or_none)
actual_lon = df['actual_longitude'].map(_to_float_or_none)
inf_lat    = df['inferred_latitude'].map(_to_float_or_none)
inf_lon    = df['inferred_longitude'].map(_to_float_or_none)

best_lat = actual_lat.where(actual_lat.notna(), inf_lat)
best_lon = actual_lon.where(actual_lon.notna(), inf_lon)

df['latitude']  = best_lat
df['longitude'] = best_lon

In [13]:
# %% Water type: conservative standardization (helper column only)
# Lowercase, collapse internal whitespace. Do not alter original column.
df['water_type_clean'] = (
    df['water_type']
      .fillna('')
      .str.lower()
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
)


In [15]:
preview(df, start=500, end=520) # rows 500–520

,id,page_number,inferred_latitude,inferred_longitude,actual_latitude,actual_longitude,location,townships_ranges_sections,watersource_name,actual_county,inferred_county,dates_of_recording,temporal_resolution,units_of_measurement,water_type,keyterms,id_page_number,latitude,longitude,water_type_clean
500,3146,259,32.9466,-110.8504,,,"Near Winkelman, Pinal County",[],SAN PEDRO RIVER NEAR WINKELMAN,"Pinal County, Arizona",,10/01/1964-09/30/1965,Daily,{'Specific_Conductance': 'micromhos at 25°C'},Stream Discharge,['Specific conductance'],3146_259,32.946600,-110.850400,stream discharge
501,1592,231,38.1833,-120.7167,,,"Below Salt Springs Dam, North Fork of Mokelumne River","['[T. 8 N., R. 16 E., Sec. 33]']",NORTH FORK OF MOKELUMNE RIVER BELOW SALT SPRINGS DAM,Calaveras County,,10/01/1932-09/30/1933,Daily,"{'Discharge': 'second-feet', 'Gage Height': 'feet'}",Stream Discharge,"['Discharge', 'Gage Height', 'Run-off']",1592_231,38.183300,-120.716700,stream discharge
502,614,235,,,37.676111,-121.264167,"Water-stage recorder in El Pescadero Grant at Durham Ferry highway bridge, 3 miles downstream from Stanislaus River and 3.4 miles northeast of Vernalis",[],"San Joaquin River near Vernalis, Calif.",San Joaquin County,,10/01/1947-09/30/1948,Daily,Second-feet,Stream Discharge,"['Discharge', 'San Joaquin River', 'Vernalis', 'stream flow', 'precipitation runoff']",614_235,37.676111,-121.264167,stream discharge
503,2513,26,,,34.0522,-118.2437,"Los Angeles, California",[],Los Angeles Rainfall,Los Angeles County,,07/01/1877-06/30/1946,Yearly,inches,Precipitation,"['Rainfall', 'Cumulative Departure', '68-year Average']",2513_26,34.052200,-118.243700,precipitation
504,70038394,49,34.6461,-120.4625,,,"Near Lompoc, California",[],Santa Ynez River Basin,Santa Barbara County,,11/13/1963-09/24/1964,Variable intervals within the range of daily to weekly measurements,"{'Temperature': '°F', 'Discharge': 'cfs (cubic feet per second)', 'Sediment Concentration': 'ppm (parts per million)', 'Suspended Sediment Discharge': 'tons per day'}",Stream Discharge,"['Suspended sediment', 'Sediment discharge', 'Sampling point', 'Discharge (cfs)']",70038394_49,34.646100,-120.462500,stream discharge
505,1305,60,34.097,-117.7462,,,"One-fourth mile west of Spadra, Pomona quadrangle",[],S.E. Hicks Well,,Los Angeles County,"Oct. 8, 1904-Jan. 4, 1911",Variable (monthly to yearly),"{'Depth_below_bench_mark': ['Feet', 'Inches']}",Groundwater,"['bored well', 'wind lift', 'domestic use']",1305_60,34.097000,-117.746200,groundwater
506,70038675,82,,,33.568889,-116.511944,"near Anza, Riverside Pines to County, Palms Highway, in SWINENW sec 18, T.7 and 9.4 miles east of Anza.","['[SWINENW sec 18, T.7]', '9.4 miles east of Anza']",PALM CANYON CREEK TRIBUTARY,San Bernardino County,,10/1969-09/1970,Daily,cubic feet per second (cfs) and inches for precipitation,Stream Discharge,"['tributary', 'stream discharge', 'culvert']",70038675_82,33.568889,-116.511944,stream discharge
507,648,413,,,39.5375,-121.3458333,"NW1 sec. 6, T. 19 N., R. 6 E.","['T. 19 N., R. 6 E., NW1 sec. 6']",South Fork Feather River at Enterprise,Plumas,,10/01/1911-09/30/1952,Daily,cubic feet per second (cfs),Stream Discharge,"['stream discharge', 'flow rates', 'gage height']",648_413,39.537500,-121.345833,stream discharge
508,56361,103,32.7806,-114.9904,32.7767,-115.573,"NORTHERLY INTERNATIONAL BOUNDARY, ABOVE MORELOS DAM, NEAR ANDRADE, CALIF.",,COLORADO RIVER,,Imperial County,10/09/1974-09/23/1975,Weekly,"{'Total_Photo_Phytoplankton_Per': 'ML)', 'Seston_Ash_Weight': 'MG/L', 'Periphyton_Biomass_Ash_Weight': 'G/SQ M', 'Periphyton_Biomass_Total_Dry_Weight': 'G/SQ M', 'Uncorrected_Periphyton_Chlorophyll_A': 'MG/SQ M', 'Chlorophyll_A': 'UG/L', 'Chlorophyll_B': 'UG/L', 'Fecal_Coli-Form_Col.Per': '100 ML)', 'Tococcic_Colonies_Per': '100 ML)', 'Strep': None, 'Total_Organic_Carbon_C': 'MG/L'}",Stream Discharge,"['WATER QUALITY DATA', 'PHYTOPLANKTON', 'SESTON', 'PERIPHYTON BIOMASS', 'CHLOROPHYLL A', 'FECAL COLIFORM']",56361_103,32

## Date parsing

In [19]:
# %% Dates parsing
YEAR_MIN = 1800
YEAR_MAX = 1980

RE_YEAR4   = re.compile(r"\b(18\d{2}|19\d{2}|2000|20\d{2})\b")
RE_MMYYYY  = re.compile(r"\b(0?[1-9]|1[0-2])[\-/](18\d{2}|19\d{2}|20\d{2})\b")
RE_YYYYMMDD= re.compile(r"\b(18\d{2}|19\d{2}|20\d{2})[\-/](0?[1-9]|1[0-2])[\-/]([0-2]?\d|3[01])\b")
RE_TWO_DIGIT_RANGE = re.compile(r"\b(18\d{2}|19\d{2})\s*[-/]\s*(\d{2})\b")

OPEN_TOKENS = (
    'present','to present','current','current year','to current year',
    'to date','t-','t/','tc','open','ongoing'
)

def _normalize_text(s: str) -> str:
    s = s.replace('\u2013','-').replace('\u2014','-')
    s = re.sub(r"[\u2212\u2012\u2015]", "-", s)
    s = s.replace(';', ',')
    s = s.replace('\u00a0', ' ')
    s = re.sub(r"\s+", " ", s)
    return s.strip()

def _expand_two_digit_end(start_year: int, end_two: int) -> int:
    base_century = start_year - (start_year % 100)
    candidate = base_century + end_two
    if end_two < (start_year % 100):
        candidate += 100
    return candidate

def parse_years(raw: Any) -> Dict[str, Any]:
    out = {'year_start': None,'year_end': None,'years_list': None,
           'date_parse_status': None,'date_parse_notes': None,'needs_date_review': False}

    if raw is None:
        out['date_parse_status'] = 'UNPARSED'; out['needs_date_review'] = True; return out

    s = str(raw).strip()
    if s == '' or s.lower() in {'not specified','-','--','n/a','na'}:
        out['date_parse_status'] = 'UNPARSED'; out['needs_date_review'] = True; return out

    norm = _normalize_text(s)
    norm_lower = norm.lower()
    open_ended = any(tok in norm_lower for tok in OPEN_TOKENS)

    years: List[int] = []

    for y in RE_YEAR4.findall(norm):
        try: years.append(int(y))
        except: pass
    for mm, yyyy in RE_MMYYYY.findall(norm):
        try: years.append(int(yyyy))
        except: pass
    for yyyy, mm, dd in RE_YYYYMMDD.findall(norm):
        try: years.append(int(yyyy))
        except: pass
    for start, end2 in RE_TWO_DIGIT_RANGE.findall(norm):
        try:
            s4 = int(start); e2 = int(end2)
            years.append(s4); years.append(_expand_two_digit_end(s4,e2))
        except: pass

    years = [y for y in years if YEAR_MIN <= y <= YEAR_MAX]
    if not years:
        out['date_parse_status'] = 'UNPARSED'; out['date_parse_notes'] = 'no_year_<=1980_found'; out['needs_date_review'] = True; return out

    years = sorted(set(years))
    out['years_list'] = years
    out['year_start'] = min(years)
    if open_ended:
        out['year_end'] = None; out['date_parse_status'] = 'OPEN_ENDED_IGNORED'; out['needs_date_review'] = True
    else:
        out['year_end'] = max(years)
        if len(years) == 1: out['date_parse_status'] = 'SINGLE_YEAR'
        elif len(years) == 2 and out['year_end'] != out['year_start']: out['date_parse_status'] = 'EXACT_RANGE'
        else: out['date_parse_status'] = 'MULTI_YEARS_OR_RANGES'
    return out

# %% Apply parser
df.rename(columns={'dates_of_recording': 'dates_of_recording_raw'}, inplace=True)
parsed = df['dates_of_recording_raw'].apply(parse_years).apply(pd.Series)
df = pd.concat([df, parsed], axis=1)

# Cast years to Int64
df['year_start'] = df['year_start'].astype('Int64')
df['year_end']   = df['year_end'].astype('Int64')

# %% Quick sanity checks
print("\nSample of parsed years:")
print(df[['dates_of_recording_raw','year_start','year_end','date_parse_status']].head(12))


Sample of parsed years:
   dates_of_recording_raw  year_start  year_start  year_start  year_start  \
0         12/1964-09/1966        1964        1964        1964        1964   
1   01/01/1912-09/30/1942        1912        1912        1912        1912   
2   10/01/1929-09/30/1930        1929        1929        1929        1929   
3      03/14/1910-09/1938        1910        1910        1910        1910   
4                                <NA>        <NA>        <NA>        <NA>   
5                                <NA>        <NA>        <NA>        <NA>   
6                                <NA>        <NA>        <NA>        <NA>   
7                                <NA>        <NA>        <NA>        <NA>   
8   10/09/1952-06/16/1950        1950        1950        1950        1950   
9   10/01/1968-09/30/1969        1968        1968        1968        1968   
10              1880-1880        1880        1880        1880        1880   
11              1884-1898        1884        1884  

In [24]:
preview(df, start=500, end=502) # rows 500–520

,id,page_number,inferred_latitude,inferred_longitude,actual_latitude,actual_longitude,location,townships_ranges_sections,watersource_name,actual_county,inferred_county,dates_of_recording_raw,temporal_resolution,units_of_measurement,water_type,keyterms,id_page_number,latitude,longitude,water_type_clean,year_start,year_end,years_list,date_parse_status,date_parse_notes,needs_date_review,year_midpoint,year_start,year_end,years_list,date_parse_status,date_parse_notes,needs_date_review,year_start,year_end,years_list,date_parse_status,date_parse_notes,needs_date_review,year_start,year_end,years_list,date_parse_status,date_parse_notes,needs_date_review
500,3146,259,32.9466,-110.8504,,,"Near Winkelman, Pinal County",[],SAN PEDRO RIVER NEAR WINKELMAN,"Pinal County, Arizona",,10/01/1964-09/30/1965,Daily,{'Specific_Conductance': 'micromhos at 25°C'},Stream Discharge,['Specific conductance'],3146_259,32.946600,-110.850400,stream discharge,1964,1965,"[1964, 1965]",EXACT_RANGE,None,False,1964.000000,1964,1965,"[1964, 1965]",EXACT_RANGE,None,False,1964,1965,"[1964, 1965]",EXACT_RANGE,None,False,1964,1965,"[1964, 1965]",EXACT_RANGE,None,False
501,1592,231,38.1833,-120.7167,,,"Below Salt Springs Dam, North Fork of Mokelumne River","['[T. 8 N., R. 16 E., Sec. 33]']",NORTH FORK OF MOKELUMNE RIVER BELOW SALT SPRINGS DAM,Calaveras County,,10/01/1932-09/30/1933,Daily,"{'Discharge': 'second-feet', 'Gage Height': 'feet'}",Stream Discharge,"['Discharge', 'Gage Height', 'Run-off']",1592_231,38.183300,-120.716700,stream discharge,1932,1933,"[1932, 1933]",EXACT_RANGE,None,False,1932.000000,1932,1933,"[1932, 1933]",EXACT_RANGE,None,False,1932,1933,"[1932, 1933]",EXACT_RANGE,None,False,1932,1933,"[1932, 1933]",EXACT_RANGE,None,False
502,614,235,,,37.676111,-121.264167,"Water-stage recorder in El Pescadero Grant at Durham Ferry highway bridge, 3 miles downstream from Stanislaus River and 3.4 miles northeast of Vernalis",[],"San Joaquin River near Vernalis, Calif.",San Joaquin County,,10/01/1947-09/30/1948,Daily,Second-feet,Stream Discharge,"['Discharge', 'San Joaquin River', 'Vernalis', 'stream flow', 'precipitation runoff']",614_235,37.676111,-121.264167,stream discharge,1947,1948,"[1947, 1948]",EXACT_RANGE,None,False,1947.000000,1947,1948,"[1947, 1948]",EXACT_RANGE,None,False,1947,1948,"[1947, 1948]",EXACT_RANGE,None,False,1947,1948,"[1947, 1948]",EXACT_RANGE,None,False


## Save outputs

In [23]:
# %% Save outputs (non-destructive)
# Master cleaned copy including helper columns
try:
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"Saved cleaned data with years -> {OUTPUT_CSV}")
except Exception as e:
    print(f"Could not save {OUTPUT_CSV}: {e}")

# Focus list for manual review
review_mask = df['needs_date_review'] | df['year_start'].isna()
review_cols = [
    'id','page_number','id_page_number','dates_of_recording_raw',
    'year_start','year_end','date_parse_status','date_parse_notes'
]
try:
    df.loc[review_mask, review_cols].to_csv(REVIEW_CSV, index=False)
    print(f"Saved rows needing date review -> {REVIEW_CSV} ({review_mask.sum():,} rows)")
except Exception as e:
    print(f"Could not save {REVIEW_CSV}: {e}")

# %% (Optional) minimal summaries to support later goals
# Counts by water_type and year_start bucket (leaves NaNs as a separate bucket)
summary = (
    df.groupby(['water_type_clean','year_start'], dropna=False)
      .size()
      .reset_index(name='n_rows')
      .sort_values(['water_type_clean','year_start'], ascending=[True, True])
)
print("\nPreview of summary by water_type_clean x year_start:")
print(summary.head(20))

# You can export this summary too, if helpful
# summary.to_csv(os.path.join(DATA_DIR, 'summary_water_type_by_year_start.csv'), index=False)

Saved cleaned data with years -> C:\Users\aeliz\Dropbox\Documents\Jupyter Notebooks\chap2_usgs\data\cleaned_metadata_with_years.csv
Could not save C:\Users\aeliz\Dropbox\Documents\Jupyter Notebooks\chap2_usgs\data\dates_needs_review.csv: "None of [Index([('n', 'e', 'e', 'd', 's', '_', 'd', 'a', 't', 'e', '_', 'r', 'e', 'v', 'i', 'e', 'w'),\n       ('n', 'e', 'e', 'd', 's', '_', 'd', 'a', 't', 'e', '_', 'r', 'e', 'v', 'i', 'e', 'w'),\n       ('n', 'e', 'e', 'd', 's', '_', 'd', 'a', 't', 'e', '_', 'r', 'e', 'v', 'i', 'e', 'w'),\n       ('n', 'e', 'e', 'd', 's', '_', 'd', 'a', 't', 'e', '_', 'r', 'e', 'v', 'i', 'e', 'w'),\n                                          ('y', 'e', 'a', 'r', '_', 's', 't', 'a', 'r', 't'),\n                                          ('y', 'e', 'a', 'r', '_', 's', 't', 'a', 'r', 't'),\n                                          ('y', 'e', 'a', 'r', '_', 's', 't', 'a', 'r', 't'),\n                                          ('y', 'e', 'a', 'r', '_', 's', 't', 'a', 'r',

ValueError: Grouper for 'year_start' not 1-dimensional